In [2]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier
import pickle

os.makedirs("../results/models",  exist_ok=True)
os.makedirs("../results/plots",   exist_ok=True)
os.makedirs("../results/metrics", exist_ok=True)

# ── Cell 2: Load Data ─────────────────────────────────────────────────────────
df = pd.read_csv("../data/resume_dataset.csv")
print(f"✅ Loaded {len(df)} resumes")

# ── Cell 3: Vectorize ─────────────────────────────────────────────────────────
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = tfidf.fit_transform(df['clean_text'].fillna('')).toarray()

# ── Cell 4: Encode category labels ────────────────────────────────────────────
le_cat = LabelEncoder()
y_cat = le_cat.fit_transform(df['Category'])
num_cat_classes = len(le_cat.classes_)
print(f"✅ Categories: {list(le_cat.classes_)}")

# ── Cell 5: Encode experience labels ──────────────────────────────────────────
le_exp = LabelEncoder()
y_exp = le_exp.fit_transform(df['experience_level'])
num_exp_classes = len(le_exp.classes_)
print(f"✅ Experience levels: {list(le_exp.classes_)}")

# ── Cell 6: Train/test split ──────────────────────────────────────────────────
X_train, X_test, yc_train, yc_test = train_test_split(X, y_cat, test_size=0.2, random_state=42, stratify=y_cat)
_, _, ye_train, ye_test             = train_test_split(X, y_exp, test_size=0.2, random_state=42, stratify=y_exp)
print(f"✅ Train: {len(X_train)} | Test: {len(X_test)}")

# ── Cell 7: Build model function ──────────────────────────────────────────────
def build_model(num_classes):
    model = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
    return model

# ── Cell 8: Train CATEGORY classifier ────────────────────────────────────────
print("\n🧠 Training Category Classifier...")
cat_model = build_model(num_cat_classes)
cat_model.fit(X_train, yc_train)
cat_model.score(X_train, yc_train)
with open("../results/models/category_classifier_final.pkl", "wb") as f:
    pickle.dump(cat_model, f)
print("✅ Category model saved!")

# ── Cell 9: Train EXPERIENCE classifier ──────────────────────────────────────
print("\n🧠 Training Experience Classifier...")
exp_model = build_model(num_exp_classes)
exp_model.fit(X_train, ye_train)
exp_model.score(X_train, ye_train)
with open("../results/models/exp_classifier_final.pkl", "wb") as f:
    pickle.dump(exp_model, f)
print("✅ Experience model saved!")

# ── Cell 11: Plot model scores ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Category scores
models = ['Train', 'Test']
cat_scores = [scores['category_train_score'], scores['category_test_score']]
axes[0].bar(models, cat_scores, color=['steelblue', 'darkorange'])
axes[0].set_title('Category Classifier Accuracy')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim([0, 1])
for i, v in enumerate(cat_scores):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center')

# Experience scores
exp_scores = [scores['experience_train_score'], scores['experience_test_score']]
axes[1].bar(models, exp_scores, color=['steelblue', 'darkorange'])
axes[1].set_title('Experience Classifier Accuracy')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim([0, 1])
for i, v in enumerate(exp_scores):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center')

plt.tight_layout()
plt.savefig("../results/plots/model_scores.png", dpi=150)
plt.show()
print("✅ Scores plot saved → results/plots/model_scores.png")

# ── Cell 10: Save training metrics ──────────────────────────────────────────
scores = {
    'category_train_score': cat_model.score(X_train, yc_train),
    'category_test_score': cat_model.score(X_test, yc_test),
    'experience_train_score': exp_model.score(X_train, ye_train),
    'experience_test_score': exp_model.score(X_test, ye_test)
}
with open("../results/metrics/model_scores.json", "w") as f:
    json.dump(scores, f)
print("✅ Model scores saved → results/metrics/model_scores.json")

# ── Cell 12: Evaluate & save reports ─────────────────────────────────────────
yc_pred = cat_model.predict(X_test)
ye_pred = exp_model.predict(X_test)

cat_report = classification_report(yc_test, yc_pred, target_names=le_cat.classes_)
exp_report = classification_report(ye_test, ye_pred, target_names=le_exp.classes_)

with open("../results/metrics/category_classification_report.txt", "w") as f:
    f.write(cat_report)
with open("../results/metrics/exp_classification_report.txt", "w") as f:
    f.write(exp_report)

print("📊 Category Report:\n", cat_report)
print("📊 Experience Report:\n", exp_report)

# ── Cell 13: Confusion matrices ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(confusion_matrix(yc_test, yc_pred), annot=True, fmt='d',
            xticklabels=le_cat.classes_, yticklabels=le_cat.classes_,
            cmap='Blues', ax=axes[0])
axes[0].set_title('Category Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].tick_params(axis='x', rotation=45)

sns.heatmap(confusion_matrix(ye_test, ye_pred), annot=True, fmt='d',
            xticklabels=le_exp.classes_, yticklabels=le_exp.classes_,
            cmap='Greens', ax=axes[1])
axes[1].set_title('Experience Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig("../results/plots/combined_confusion_matrices.png", dpi=150)
plt.show()
print("✅ Confusion matrices saved!")

FileNotFoundError: [Errno 2] No such file or directory: '../data/resume_dataset.csv'